<a href="https://colab.research.google.com/github/ValentinaEmili/Texture-synthesis/blob/main/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import time
import numpy as np
import math

![image](https://peterbloem.nl/files/transformers/transformer-block.svg)

Causal Multi-Head Self-Attention: a multi-head masked self-attention layer with a projection at the end. Implementation from scratch of `torch.nn.MultiheadAttention()`

In [ ]:
class MultiHead_SelfAttention(nn.Module):
    def __init__(self, embed_dim, heads, dropout=0.1):
        super().__init__()
        assert embed_dim % heads == 0, "embed_dim must be divisible by heads"
        self.embed_dim = embed_dim
        self.heads = heads
        self.head_dim = embed_dim // heads

        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)

        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        b, t, m = x.size()  # (batch_size, seq_length, embed_dim)
        r = self.heads

        queries = self.W_q(x).view(b, t, r, self.head_dim)
        keys = self.W_k(x).view(b, t, r, self.head_dim)
        values = self.W_v(x).view(b, t, r, self.head_dim)

        causal_mask = torch.tril(torch.ones(t, t, device=x.device)).view(1, 1, t, t)

        w = torch.einsum('btrd,bfrd->brtf', queries, keys) / math.sqrt(self.head_dim)
        w = w.masked_fill(causal_mask == 0, float('-inf'))
        w = F.softmax(w, dim=-1)
        w = self.attn_drop(w)

        y = torch.einsum('brtf,bfrd->btrd', w, values)
        y = y.contiguous().view(b, t, m)
        y = self.proj(y)
        y = self.resid_drop(y)
        return y

In [ ]:
class MLP(nn.Module):
    def __init__(self, embed_dim, dropout=0.1):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class Block(nn.Module):
    def __init__(self, embed_dim, heads, dropout=0.1):
        super().__init__()
        self.attn = MultiHead_SelfAttention(embed_dim, heads, dropout)
        self.ln_1 = nn.LayerNorm(embed_dim)
        self.ln_2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, dropout)

    def forward(self, x):
      x = x + self.attn(self.ln_1(x))
      x = x + self.mlp(self.ln_2(x))
      return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self, num_embeddings, seq_len, embed_dim=256, heads=8, n_layers=6, dropout=0.1):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.seq_len = seq_len

        vocab_size = num_embeddings + 1 # codebook indices + BOS token index

        # embeddings
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Parameter(torch.zeros(1, seq_len, embed_dim))
        self.dropout = nn.Dropout(dropout)

        # transformer blocks
        self.blocks = nn.ModuleList([Block(embed_dim, heads, dropout) for _ in range(n_layers)])

        # decoder head
        self.ln = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_embeddings, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
          module.weight.data.normal_(mean=0.0, std=0.02)
          if module.bias is not None:
            module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
          module.weight.data.normal_(mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
          module.bias.data.zero_()
          module.weight.data.fill_(1.0)

    def forward(self, idx):
        b, t = idx.shape

        token_embeddings  = self.token_emb(idx)     # (b, t, embed_dim)
        pos_embeddings    = self.pos_emb[:, :t, :]  # (1, t, embed_dim)

        x = self.dropout(token_embeddings + pos_embeddings)

        for i, block in enumerate(self.blocks):
          x = block(x)

        x = self.ln(x)
        logits = self.head(x)                       # (b, t, num_embeddings)
        return logits